In [ ]:
import polars as pl
from pathlib import Path
import shutil

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
#### CHANGE THIS NAME ####
brms_run_name = "brms_polarity_test"
##########################

groups = [
    "fungi_mit",
    "metazoans_mit",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
    "green_algae_mit",
    "green_algae_plt",
]
polarity_2bin = {
    "++": "same",
    "--": "same",
    "+-": "opp",
    "-+": "opp",
}
polarity_3bin = {
    "++": "same",
    "--": "same",
    "+-": "conv",
    "-+": "div",
}

In [ ]:
trna_labels = {
    "TA",
    "TASX",
    "ASX",
    "TC",
    "TD",
    "TE",
    "TER",
    "TF",
    "TFM",
    "TG",
    "TGLX",
    "TH",
    "TI",
    "TK",
    "TL1|2",
    "TM",
    "TN",
    "TP",
    "TQ",
    "TR",
    "TRMR",
    "TRNX",
    "TS1|2",
    "TSEC",
    "TSUP",
    "TT",
    "TU",
    "TV",
    "TW",
    "TX",
    "TXLE",
    "XLE",
    "TY",
}

rrna_labels = {
    "RNR12",
    "RNR15",
    "RNR16",
    "RNR18",
    "RNR21",
    "RNR23",
    "RNR26",
    "RNR28",
    "RNR4",
    "RNR4.5",
    "RNR5",
    "RNR9",
    "RNRL",
    "RNRS",
    "RRNA",
    "RRN3",
    "RRN7",
    "RRNF",
    "RRF",
}

intronic_labels = {
    "GIY-YIG",
    "I-END",
    "I-MAT",
    "I-RT",
    "LAGLIDADG",
    "MAT",
    "NAT",
    "RT",
    "LAG-END",
    "MATK",
    "MATR",
    "END-LAG",
}

In [ ]:
# Output columns for <brms_run_name>/filtered_3bin.tsv (brm's input data)
brms_cols = [
    "AN",
    "ID",
    "polarity_3bin",
    "igr_len",
    "flanking_types",
    "ncbi_taxid",
    "log10_igr_len",
    "genome_length",
    "log10_genome_length",
    "log10_geom_mean_flanking_gene_len",
    "log10_min_flanking_gene_len",
    "log10_max_flanking_gene_len",
    "sup_len",
    "sdw_len",
    "ncbi_name",
]

In [ ]:
all_data = {}
for g in groups:
    gdir = BASE / g
    igr = pl.read_csv(gdir / f"{g}_summary_igr.tsv", separator="\t")
    tsv = (
        pl.read_csv(gdir / f"{g}.tsv", separator="\t")
        .select(["AN", "ncbi_taxid", "Genome_length", "ncbi_name"])
        .rename({"Genome_length": "genome_length"})
    )

    igr = igr.join(tsv, on="AN", how="inner")

    igr = (
        igr.with_columns(
            pl.col("source_up")
            .str.split_exact("|", 5)
            .struct.rename_fields(
                [
                    "sup_an",
                    "sup_type",
                    "sup_start",
                    "sup_end",
                    "sup_polarity",
                    "sup_gff_id",
                ]
            )
            .alias("parts")
        )
        .unnest("parts")
        .with_columns(
            pl.col("sup_start").cast(pl.Int64),
            pl.col("sup_end").cast(pl.Int64),
        )
        .drop(["sup_an", "sup_type", "sup_gff_id"])
    )

    igr = (
        igr.with_columns(
            pl.col("source_dw")
            .str.split_exact("|", 5)
            .struct.rename_fields(
                [
                    "sdw_an",
                    "sdw_type",
                    "sdw_start",
                    "sdw_end",
                    "sdw_polarity",
                    "sdw_gff_id",
                ]
            )
            .alias("parts")
        )
        .unnest("parts")
        .with_columns(
            pl.col("sdw_start").cast(pl.Int64),
            pl.col("sdw_end").cast(pl.Int64),
        )
        .drop(["sdw_an", "sdw_type", "sdw_gff_id"])
    )

    igr = igr.with_columns(
        pl.col("Polarity").replace_strict(polarity_3bin).alias("polarity_3bin"),
        pl.col("Polarity").replace_strict(polarity_2bin).alias("polarity_2bin"),
    )

    igr = igr.with_columns(
        (pl.col("sup_end") - pl.col("sup_start") + 1).alias("sup_len"),
        (pl.col("sdw_end") - pl.col("sdw_start") + 1).alias("sdw_len"),
    )
    igr = igr.with_columns(
        pl.col("sup_len").fill_null(pl.col("sdw_len")).alias("sup_len"),
        pl.col("sdw_len").fill_null(pl.col("sup_len")).alias("sdw_len"),
    )
    igr = igr.with_columns(
        ((pl.col("sup_len") + pl.col("sdw_len")) / 2).alias("mean_flanking_gene_len"),
        ((pl.col("sup_len").log10() + pl.col("sdw_len").log10()) / 2).alias(
            "log10_geom_mean_flanking_gene_len"
        ),
        (pl.min_horizontal("sup_len", "sdw_len").log10()).alias(
            "log10_min_flanking_gene_len"
        ),
        (pl.max_horizontal("sup_len", "sdw_len").log10()).alias(
            "log10_max_flanking_gene_len"
        ),
    )
    igr = igr.with_columns(
        pl.col("mean_flanking_gene_len").log10().alias("log10_mean_flanking_gene_len"),
        pl.col("Length").log10().alias("log10_igr_len"),
        pl.col("genome_length").log10().alias("log10_genome_length"),
    )
    log10_mean_flanking_gene_len_mean = igr["log10_mean_flanking_gene_len"].mean()
    igr = igr.with_columns(
        (
            pl.col("log10_mean_flanking_gene_len") - log10_mean_flanking_gene_len_mean
        ).alias("centered_log10_mean_flanking_gene_len"),
    )
    igr = igr.rename({"Length": "igr_len"})

    igr = igr.with_columns(
        pl.col("UP").str.replace_all(r"^MIT-|^PLT-", "").alias("label_up"),
        pl.col("DOWN").str.replace_all(r"^MIT-|^PLT-", "").alias("label_dw"),
    )
    igr = igr.with_columns(
        pl.when(pl.col("label_up").is_in(trna_labels))
        .then(pl.lit("trna"))
        .when(pl.col("label_up").is_in(rrna_labels))
        .then(pl.lit("rrna"))
        # .when(pl.col("label_up").is_in(intronic_labels)).then(pl.lit("intronic"))
        .otherwise(pl.lit("gene")).alias("type_up"),
        pl.when(pl.col("label_dw").is_in(trna_labels))
        .then(pl.lit("trna"))
        .when(pl.col("label_dw").is_in(rrna_labels))
        .then(pl.lit("rrna"))
        # .when(pl.col("label_dw").is_in(intronic_labels)).then(pl.lit("intronic"))
        .otherwise(pl.lit("gene")).alias("type_dw"),
    )
    igr = igr.with_columns(
        pl.when(pl.col("type_up") <= pl.col("type_dw"))
        .then(pl.col("type_up") + "-" + pl.col("type_dw"))
        .otherwise(pl.col("type_dw") + "-" + pl.col("type_up"))
        .alias("flanking_types")
    )

    all_data[g] = igr

In [ ]:
for g in groups:
    data = all_data[g]

    gdir = BASE / g
    brms_folder = gdir / brms_run_name
    brms_folder.mkdir(exist_ok=True)

    unique_flanking_types = data["flanking_types"].unique()
    # print(f"Unique flanking types for {g}: {unique_flanking_types.to_list()}")

    tree = BASE / g / "tree.nwk"
    shutil.copy(tree, brms_folder / "tree.nwk")

    data = data.select(brms_cols)

    numeric_required = [
        "log10_igr_len",
        "log10_geom_mean_flanking_gene_len",
        "log10_min_flanking_gene_len",
        "log10_max_flanking_gene_len",
    ]
    categorical_required = ["ncbi_taxid", "polarity_3bin"]

    for col in numeric_required:
        bad = data.filter(
            pl.col(col).is_null() | pl.col(col).is_nan() | pl.col(col).is_infinite()
        )
        if bad.height > 0:
            print(f"  [{g}] {col}: {bad.height} problematic rows")
            print(bad)

    for col in categorical_required:
        bad = data.filter(pl.col(col).is_null())
        if bad.height > 0:
            print(f"  [{g}] {col}: {bad.height} null rows")
            print(bad)

    # round numerical columns to 4 decimal places
    for col in data.columns:
        if data[col].dtype in [pl.Float32, pl.Float64]:
            data = data.with_columns(pl.col(col).round(4))

    # print(data["polarity_3bin"].value_counts())
    # print(data["ncbi_taxid"].n_unique(), "taxa")
    data.write_csv(brms_folder / "filtered_3bin.tsv", separator="\t")

    print(f"Prepared data for {g}")

In [ ]:
N = 1
ans_to_keep = {}
print(f"\nN={N}")
for g in ["metazoans_mit", "plants_plt"]:
    gdir = BASE / g
    tsv = pl.read_csv(gdir / f"{g}.tsv", separator="\t").select(["AN", "ncbi_name"])

    tsv = tsv.with_columns(
        pl.col("ncbi_name").str.split_exact(" ", 1).struct[0].alias("genus")
    )

    n_per_genus = (
        tsv.group_by("genus")
        .agg(pl.col("AN").head(N))
        .explode("AN")
        .select(["AN", "genus"])
    )
    ans_to_keep[g] = n_per_genus["AN"].to_list()

    data = all_data[g]
    data = data.filter(pl.col("AN").is_in(ans_to_keep[g]))
    print(f"{g}: {data.height} rows after filtering to {N} per genus")
    all_data[g] = data

    brms_folder = gdir / brms_run_name
    data.write_csv(brms_folder / "filtered_3bin.tsv", separator="\t")

    print(
        f"{g}: ans: {n_per_genus["AN"].n_unique()}, genera {n_per_genus["genus"].n_unique()}"
    )